## Partial Observation + And-Or-Graph Search


In [3]:
import tkinter as tk
from tkinter import messagebox, ttk
from copy import deepcopy
from collections import deque
from itertools import permutations
import threading
import time
import queue as Queue
import random
import inspect

# ==============================================================
# PARTIAL OBSERVATION — ALGORITHM
# ==============================================================

PO_MOVES = {
    "UP"   : (-1,  0),
    "DOWN" : ( 1,  0),
    "LEFT" : ( 0, -1),
    "RIGHT": ( 0,  1),
}


def po_apply_action(state, action):

    dx, dy = PO_MOVES[action]

    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                nx, ny = i + dx, j + dy
                if 0 <= nx < 3 and 0 <= ny < 3:
                    new_state = deepcopy(state)
                    new_state[i][j], new_state[nx][ny] = \
                        new_state[nx][ny], new_state[i][j]
                    return new_state, True
                else:
                    return deepcopy(state), False

    return deepcopy(state), False


def po_po_state_to_tuple(state):
    return tuple(state[i][j] for i in range(3) for j in range(3))


def po_po_belief_to_key(belief):
    return frozenset(po_po_state_to_tuple(s) for s in belief)


def random_full_state():
    nums = list(range(9))
    random.shuffle(nums)
    return [[nums[i*3+j] for j in range(3)] for i in range(3)]


def mask_state(full_state, n_hidden=2):

    positions = [(i, j) for i in range(3) for j in range(3)]
    hidden = random.sample(positions, n_hidden)

    masked = deepcopy(full_state)
    for (i, j) in hidden:
        masked[i][j] = None

    return masked, hidden


def generate_belief(masked_grid, user_values, max_states=50):

    known_values = set()
    hidden_cells  = []

    for i in range(3):
        for j in range(3):
            if masked_grid[i][j] is None:
                hidden_cells.append((i, j))
            else:
                known_values.add(masked_grid[i][j])

    missing_values = set(range(9)) - known_values

    belief = []

    candidate_lists = []
    for cell in hidden_cells:
        if cell in user_values and user_values[cell]:
            vals = [v for v in user_values[cell] if v in missing_values]
            if not vals:
                vals = list(missing_values)
        else:
            vals = list(missing_values)
        candidate_lists.append(vals)

    seen = set()

    def backtrack(idx, used, current_assignment):
        if len(belief) >= max_states:
            return
        if idx == len(hidden_cells):
            if used == missing_values:
                grid = deepcopy(masked_grid)
                for cell, val in zip(hidden_cells, current_assignment):
                    grid[cell[0]][cell[1]] = val
                key = po_po_state_to_tuple(grid)
                if key not in seen:
                    seen.add(key)
                    belief.append(grid)
            return

        for v in candidate_lists[idx]:
            if v in used:
                continue
            backtrack(idx + 1, used | {v}, current_assignment + [v])
            if len(belief) >= max_states:
                return

    backtrack(0, set(), [])

    return belief


# ==========================================
# BFS TRÊN BELIEF-STATE -> BELIEF-GOAL
#
# FIX: node đã đạt goal trong belief giữ nguyên, không áp action lên nó.
# ==========================================

def belief_state_bfs_partial(belief_start, belief_goal,
                              stop_event=None, log_queue=None,
                              max_nodes=5000):

    goal_keys = set(po_po_state_to_tuple(s) for s in belief_goal)

    def all_reach_goal(belief):
        return all(po_po_state_to_tuple(s) in goal_keys for s in belief)

    initial_key = po_po_belief_to_key(belief_start)

    queue = deque()
    queue.append((belief_start, []))

    visited = {initial_key}
    expanded = 0

    if all_reach_goal(belief_start):
        return "ALREADY_GOAL"

    while queue:

        if stop_event and stop_event.is_set():
            return "CANCELLED"

        if expanded >= max_nodes:
            if log_queue is not None:
                log_queue.put(f"__LIMIT__:{expanded}")
            return "LIMIT_REACHED"

        belief, path = queue.popleft()

        for action in PO_MOVES:

            new_belief = []
            for s in belief:
                # ── FIX: node đã đạt goal → giữ nguyên ──────────────────
                if po_po_state_to_tuple(s) in goal_keys:
                    new_belief.append(deepcopy(s))
                else:
                    new_s, _ = po_apply_action(s, action)
                    new_belief.append(new_s)
                # ─────────────────────────────────────────────────────────

            new_key = po_po_belief_to_key(new_belief)

            if new_key in visited:
                continue

            visited.add(new_key)
            expanded += 1

            new_path = path + [(deepcopy(new_belief), action)]

            if log_queue is not None:
                row_strs = []
                for idx, s in enumerate(new_belief):
                    at_goal = "✓" if po_po_state_to_tuple(s) in goal_keys else " "
                    rows = " | ".join(
                        " ".join(str(v) if v != 0 else "_" for v in s[r])
                        for r in range(3)
                    )
                    row_strs.append(f"  B{idx+1}[{at_goal}]: {rows}")
                entry = (
                    f"[{expanded}] ACTION: {action} (belief size={len(new_belief)})\n"
                    + "\n".join(row_strs)
                    + "\n" + "-"*35 + "\n"
                )
                log_queue.put(entry)

            if all_reach_goal(new_belief):
                return new_path

            queue.append((new_belief, new_path))

    return "NO_SOLUTION"

# ==============================================================
# AND-OR-GRAPH-SEARCH — ALGORITHM
# ==============================================================

AOR_MOVES = {
    "UP"   : (-1,  0),
    "DOWN" : ( 1,  0),
    "LEFT" : ( 0, -1),
    "RIGHT": ( 0,  1),
}

FAILURE = "failure"


def actions(state):

    valid = []
    x = y = 0

    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                x, y = i, j

    for action, (dx, dy) in AOR_MOVES.items():
        nx, ny = x + dx, y + dy
        if 0 <= nx < 3 and 0 <= ny < 3:
            valid.append(action)

    return valid


def results(state, action):

    dx, dy = AOR_MOVES[action]
    x = y = 0

    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                x, y = i, j

    nx, ny = x + dx, y + dy
    new_state = deepcopy(state)
    new_state[x][y], new_state[nx][ny] = \
        new_state[nx][ny], new_state[x][y]

    return [new_state]


def aos_aos_state_to_tuple(state):
    return tuple(state[i][j] for i in range(3) for j in range(3))


def _and_or_search_once(start, goal, depth_limit, stop_event,
                         log_queue, expanded_counter, max_expanded):

    def goal_test(state):
        return state == goal

    def or_search(state, path):

        if stop_event and stop_event.is_set():
            return "STOPPED"

        if expanded_counter[0] >= max_expanded:
            return "STOPPED"

        expanded_counter[0] += 1

        if log_queue is not None and expanded_counter[0] % 200 == 0:
            rows = " | ".join(
                " ".join(str(v) if v != 0 else "_" for v in state[r])
                for r in range(3)
            )
            log_queue.put(
                f"[{expanded_counter[0]}] OR-SEARCH state: {rows} "
                f"(depth={len(path)})\n"
            )

        if goal_test(state):
            return []

        if aos_aos_state_to_tuple(state) in path:
            return FAILURE

        if len(path) >= depth_limit:
            return FAILURE

        for action in actions(state):

            result_states = results(state, action)

            plan = and_search(
                result_states, path + [aos_aos_state_to_tuple(state)]
            )

            if plan == "STOPPED":
                return "STOPPED"

            if plan != FAILURE:
                return [action, plan]

        return FAILURE

    def and_search(states, path):

        plans = {}

        for s in states:

            plan_s = or_search(s, path)

            if plan_s == "STOPPED":
                return "STOPPED"

            if plan_s == FAILURE:
                return FAILURE

            plans[aos_aos_state_to_tuple(s)] = plan_s

        return plans

    plan = or_search(start, [])
    return plan


def and_or_graph_search(start, goal, stop_event=None,
                         log_queue=None, max_depth=30,
                         max_expanded=200000,
                         expanded_counter=None):

    if expanded_counter is None:
        expanded_counter = [0]

    for depth_limit in range(1, max_depth + 1):

        if stop_event and stop_event.is_set():
            return "failure", expanded_counter[0]

        if log_queue is not None:
            log_queue.put(f"-- Thu depth_limit = {depth_limit} --\n")

        plan = _and_or_search_once(
            start, goal, depth_limit, stop_event,
            log_queue, expanded_counter, max_expanded
        )

        if plan == "STOPPED":
            if expanded_counter[0] >= max_expanded:
                return "LIMIT_REACHED", expanded_counter[0]
            return "failure", expanded_counter[0]

        if plan != FAILURE:
            return plan, expanded_counter[0]

    return "failure", expanded_counter[0]


def flatten_plan(plan, start):

    sequence = []
    state = deepcopy(start)

    while True:

        if plan == FAILURE or plan is None:
            break

        if plan == []:
            break

        action, sub_plans = plan

        sequence.append((deepcopy(state), action))

        new_state = results(state, action)[0]
        key = aos_aos_state_to_tuple(new_state)

        if key not in sub_plans:
            break

        plan  = sub_plans[key]
        state = new_state

    sequence.append((deepcopy(state), "GOAL"))

    return sequence


# ==============================================================
# BELIEF-STATE BFS (3 state cố định)
#
# FIX: node đã đạt goal trong belief giữ nguyên, không áp action lên nó.
# ==============================================================

from collections import deque

BFS_MOVES = {
    "UP"   : (-1,  0),
    "DOWN" : ( 1,  0),
    "LEFT" : ( 0, -1),
    "RIGHT": ( 0,  1),
}


def bfs_apply_action(state, action):
    dx, dy = BFS_MOVES[action]
    for i in range(3):
        for j in range(3):
            if state[i][j] == 0:
                nx, ny = i + dx, j + dy
                if 0 <= nx < 3 and 0 <= ny < 3:
                    new_state = deepcopy(state)
                    new_state[i][j], new_state[nx][ny] = \
                        new_state[nx][ny], new_state[i][j]
                    return new_state, True
                else:
                    return deepcopy(state), False
    return deepcopy(state), False


def bfs_state_to_tuple(state):
    return tuple(state[i][j] for i in range(3) for j in range(3))


def bfs_belief_to_key(belief):
    return tuple(bfs_state_to_tuple(s) for s in belief)


def belief_state_bfs(starts, goal, stop_event=None, log_queue=None, max_nodes=5000):

    initial_belief = list(starts)
    initial_key    = bfs_belief_to_key(initial_belief)

    bfs_queue = deque()
    bfs_queue.append((initial_belief, []))

    visited  = {initial_key}
    expanded = 0

    while bfs_queue:

        if stop_event and stop_event.is_set():
            return "CANCELLED"

        if expanded >= max_nodes:
            if log_queue is not None:
                log_queue.put(f"__LIMIT__:{expanded}")
            return "LIMIT_REACHED"

        belief, path = bfs_queue.popleft()

        for action in BFS_MOVES:

            new_belief = []
            for s in belief:
                # ── FIX: node đã đạt goal → giữ nguyên ──────────────────
                if s == goal:
                    new_belief.append(deepcopy(s))
                else:
                    new_s, _ = bfs_apply_action(s, action)
                    new_belief.append(new_s)
                # ─────────────────────────────────────────────────────────

            new_key = bfs_belief_to_key(new_belief)

            if new_key in visited:
                continue

            visited.add(new_key)
            expanded += 1

            new_path = path + [(deepcopy(new_belief), action)]

            if log_queue is not None:
                row_strs = []
                for b in range(3):
                    at_goal = "✓" if new_belief[b] == goal else " "
                    rows = " | ".join(
                        " ".join(str(v) if v != 0 else "_" for v in new_belief[b][r])
                        for r in range(3)
                    )
                    row_strs.append(f"  S{b+1}[{at_goal}]: {rows}")
                entry = (
                    f"[{expanded}] ACTION: {action}\n"
                    + "\n".join(row_strs)
                    + "\n" + "-"*35 + "\n"
                )
                log_queue.put(entry)

            if all(s == goal for s in new_belief):
                return new_path

            bfs_queue.append((new_belief, new_path))

    return "NO_SOLUTION"


# ==============================================================
# GUI
# ==============================================================

ALGO_BELIEF_BFS = "Belief-State BFS (3 states co dinh)"
ALGO_PARTIAL    = "Partial Observation Search"
ALGO_AND_OR     = "AND-OR-GRAPH-SEARCH"

ALGO_LIST = [ALGO_BELIEF_BFS, ALGO_PARTIAL, ALGO_AND_OR]


class MultiAlgoGUI:

    def __init__(self, root):

        self.root = root
        self.root.title("8-Puzzle - Multi Algorithm")
        self.root.geometry("1700x950")

        self.default_goal = [
            [1, 2, 3],
            [4, 5, 6],
            [7, 8, 0]
        ]

        self.default_starts = [
            [[1, 2, 3], [4, 5, 6], [7, 0, 8]],
            [[1, 2, 3], [4, 5, 6], [0, 7, 8]],
            [[1, 2, 3], [4, 0, 6], [7, 5, 8]],
        ]

        self.path        = []
        self.index       = 0
        self.log_blocks  = []
        self.goal_state  = self.default_goal
        self._running    = False
        self._stop_event = threading.Event()
        self._log_queue  = Queue.Queue()
        self._poll_id    = None

        self.po_masked_start = None
        self.po_masked_goal  = None
        self.po_hidden_start = []
        self.po_hidden_goal  = []
        self.po_entry_widgets_start = {}
        self.po_entry_widgets_goal  = {}

        self._build_ui()
        self._switch_algo()

    def _build_ui(self):

        top = tk.Frame(self.root)
        top.pack(fill="x", pady=8)

        tk.Label(
            top, text="8-Puzzle - Multi Algorithm",
            font=("Arial", 20, "bold")
        ).pack(side="left", padx=10)

        tk.Label(top, text="Thuat toan:", font=("Arial", 12)).pack(side="left", padx=(30, 4))

        self.algo_var = tk.StringVar(value=ALGO_LIST[0])
        self.algo_combo = ttk.Combobox(
            top, textvariable=self.algo_var,
            values=ALGO_LIST, state="readonly",
            font=("Arial", 11), width=35
        )
        self.algo_combo.pack(side="left")
        self.algo_combo.bind("<<ComboboxSelected>>", lambda e: self._switch_algo())

        main = tk.Frame(self.root)
        main.pack(fill="both", expand=True)

        self.left = tk.Frame(main, width=520)
        self.left.pack(side="left", fill="y", padx=10, pady=5)

        right = tk.Frame(main)
        right.pack(side="left", fill="both", expand=True, padx=6, pady=5)

        self.display_frame = tk.LabelFrame(
            right, text="Current State / Belief",
            font=("Arial", 12, "bold")
        )
        self.display_frame.pack(fill="x", padx=4, pady=(4, 8))

        log_frame = tk.LabelFrame(
            right, text="Log",
            font=("Arial", 12, "bold")
        )
        log_frame.pack(fill="both", expand=True, padx=4, pady=(0, 4))

        scrollbar = tk.Scrollbar(log_frame)
        scrollbar.pack(side="right", fill="y")

        self.log = tk.Text(
            log_frame, font=("Consolas", 11),
            yscrollcommand=scrollbar.set
        )
        self.log.pack(fill="both", expand=True)
        scrollbar.config(command=self.log.yview)

    def _switch_algo(self):

        self._stop_event.set()
        self._running = False
        if self._poll_id is not None:
            try:
                self.root.after_cancel(self._poll_id)
            except Exception:
                pass
            self._poll_id = None

        for w in self.left.winfo_children():
            w.destroy()

        for w in self.display_frame.winfo_children():
            w.destroy()

        self.path       = []
        self.index      = 0
        self.log_blocks = []
        self.log.delete("1.0", tk.END)

        algo = self.algo_var.get()

        if algo == ALGO_BELIEF_BFS:
            self._build_belief_bfs_ui()
        elif algo == ALGO_PARTIAL:
            self._build_partial_ui()
        elif algo == ALGO_AND_OR:
            self._build_and_or_ui()

    # ==============================================================
    # 1) BELIEF-STATE BFS (3 states)
    # ==============================================================

    def _build_belief_bfs_ui(self):

        left = self.left

        starts_frame = tk.LabelFrame(
            left, text="Start States (Belief)", font=("Arial", 11, "bold")
        )
        starts_frame.pack(pady=6)

        self.bfs_start_entries = []
        for b in range(3):
            sf = tk.LabelFrame(starts_frame, text=f"State {b+1}", font=("Arial", 10))
            sf.grid(row=0, column=b, padx=8, pady=4)
            grid_e = []
            for i in range(3):
                row_e = []
                for j in range(3):
                    e = tk.Entry(sf, width=3, justify="center", font=("Arial", 18))
                    e.grid(row=i, column=j, padx=2, pady=2)
                    e.insert(0, str(self.default_starts[b][i][j]))
                    row_e.append(e)
                grid_e.append(row_e)
            self.bfs_start_entries.append(grid_e)

        goal_frame = tk.LabelFrame(left, text="Goal State", font=("Arial", 11, "bold"))
        goal_frame.pack(pady=6)

        self.bfs_goal_entries = []
        for i in range(3):
            row_e = []
            for j in range(3):
                e = tk.Entry(goal_frame, width=3, justify="center", font=("Arial", 18))
                e.grid(row=i, column=j, padx=2, pady=2)
                e.insert(0, str(self.default_goal[i][j]))
                row_e.append(e)
            self.bfs_goal_entries.append(row_e)

        tk.Button(
            left, text="Random Start & Goal", font=("Arial", 11),
            bg="#7c3aed", fg="white", command=self._bfs_randomize
        ).pack(fill="x", padx=4, pady=(0, 6))

        self._build_common_controls(run_callback=self._run_belief_bfs)

        COLORS = ["#2563eb", "#7c3aed", "#0891b2"]
        self.belief_cells = []
        for b in range(3):
            bf = tk.LabelFrame(
                self.display_frame, text=f"State {b+1}",
                font=("Arial", 10, "bold"), fg=COLORS[b]
            )
            bf.grid(row=0, column=b, padx=18, pady=8)
            grid_c = []
            for i in range(3):
                row_c = []
                for j in range(3):
                    lbl = tk.Label(
                        bf, width=3, height=1, font=("Arial", 20, "bold"),
                        bg=COLORS[b], fg="white", relief="solid"
                    )
                    lbl.grid(row=i, column=j, padx=3, pady=3)
                    row_c.append(lbl)
                grid_c.append(row_c)
            self.belief_cells.append(grid_c)

        starts = [deepcopy(s) for s in self.default_starts]
        self._draw_belief_generic(self.belief_cells, starts)

    def _bfs_read_input(self):
        try:
            starts = []
            for b in range(3):
                grid = []
                for i in range(3):
                    row = []
                    for j in range(3):
                        row.append(int(self.bfs_start_entries[b][i][j].get()))
                    grid.append(row)
                starts.append(grid)
            goal = []
            for i in range(3):
                row = []
                for j in range(3):
                    row.append(int(self.bfs_goal_entries[i][j].get()))
                goal.append(row)
            return starts, goal
        except Exception:
            return None, None

    def _bfs_is_solvable(self, state, goal):
        def inv_count(s):
            flat = [v for row in s for v in row if v != 0]
            count = 0
            for i in range(len(flat)):
                for j in range(i+1, len(flat)):
                    if flat[i] > flat[j]:
                        count += 1
            return count
        return (inv_count(state) % 2) == (inv_count(goal) % 2)

    def _bfs_randomize(self):
        nums = list(range(9))
        random.shuffle(nums)
        goal = [[nums[i*3+j] for j in range(3)] for i in range(3)]

        starts = []
        attempts = 0
        while len(starts) < 3 and attempts < 10000:
            attempts += 1
            random.shuffle(nums)
            cand = [[nums[i*3+j] for j in range(3)] for i in range(3)]
            if cand == goal:
                continue
            if not self._bfs_is_solvable(cand, goal):
                continue
            if cand in starts:
                continue
            starts.append(cand)

        if len(starts) < 3:
            return

        for b in range(3):
            for i in range(3):
                for j in range(3):
                    self.bfs_start_entries[b][i][j].delete(0, tk.END)
                    self.bfs_start_entries[b][i][j].insert(0, str(starts[b][i][j]))
        for i in range(3):
            for j in range(3):
                self.bfs_goal_entries[i][j].delete(0, tk.END)
                self.bfs_goal_entries[i][j].insert(0, str(goal[i][j]))

        self._draw_belief_generic(self.belief_cells, starts)
        self.log.delete("1.0", tk.END)
        self.path = []
        self.index = 0
        self.log_blocks = []
        self.info["text"] = ""
        self.time_label["text"] = "Da sinh ngau nhien"

    def _run_belief_bfs(self):

        starts, goal = self._bfs_read_input()
        if starts is None:
            messagebox.showerror("Error", "Invalid Input")
            return

        self.goal_state = goal
        self._start_search(
            target_func=belief_state_bfs,
            args=(starts, goal),
            on_done=lambda result, elapsed: self._on_done_belief_bfs(result, starts, goal, elapsed)
        )

    def _on_done_belief_bfs(self, result, starts, goal, elapsed):

        if result == "LIMIT_REACHED":
            self.log.insert(tk.END, "\n--- Vuot gioi han node, chua tim duoc solution. ---\n")
            self.log.insert(tk.END, "Hay tang 'Gioi han node' hoac chon Start/Goal khac.\n")
            self.log.see(tk.END)
            return

        if result == "NO_SOLUTION":
            self.log.insert(tk.END, "\n--- Khong tim duoc solution (da duyet het khong gian)! ---\n")
            self.log.see(tk.END)
            return

        if result == "CANCELLED" or not result:
            self.log.insert(tk.END, "\n--- Da huy / khong co ket qua ---\n")
            self.log.see(tk.END)
            return

        self.log.delete("1.0", tk.END)
        initial_belief = [deepcopy(s) for s in starts]
        self.path  = [(initial_belief, "START")] + result
        self.index = 0

        self.log_blocks = []
        header = (
            "===== BELIEF-STATE BFS =====\n\n"
            f"So trang thai ban dau : {len(starts)}\n"
            f"So buoc tim duoc      : {len(self.path) - 1}\n\n"
        )
        for i, (belief, action) in enumerate(self.path):
            block = ""
            if i == 0:
                block += header
            block += f"STEP {i}\n"
            block += f"ACTION : {action}\n"
            for b in range(len(belief)):
                at_goal = " [GOAL]" if belief[b] == goal else ""
                rows = " | ".join(
                    " ".join(str(v) if v != 0 else "_" for v in belief[b][r])
                    for r in range(3)
                )
                block += f"  State {b+1}{at_goal}: {rows}\n"
            at_goal_list = [b for b in range(len(belief)) if belief[b] == goal]
            if at_goal_list:
                block += "  OK Dat goal: State " + ", ".join(str(b+1) for b in at_goal_list) + "\n"
            block += "-" * 35 + "\n"
            self.log_blocks.append(block)

        self._render_log_up_to(self.index)
        self._show_belief_bfs_step()

    def _show_belief_bfs_step(self):

        if not self.path:
            return

        belief, action = self.path[self.index]
        prev_belief = self.path[self.index-1][0] if self.index > 0 else None

        self._draw_belief_generic(self.belief_cells, belief, prev_belief)

        n_goal = sum(1 for s in belief if s == self.goal_state)

        self.info["text"] = (
            f"STEP   : {self.index}/{len(self.path)-1}\n"
            f"ACTION : {action}\n"
            f"At goal: {n_goal}/{len(belief)}"
        )

    # ==============================================================
    # 2) PARTIAL OBSERVATION SEARCH
    # ==============================================================

    def _build_partial_ui(self):

        left = self.left

        info_text = (
            "He thong tu sinh 1 state Start va 1 state Goal\n"
            "ngau nhien, sau do CHE DI 2 o moi luoi.\n"
            "Hay nhap gia tri cho cac o trong (mau do)."
        )
        tk.Label(
            left, text=info_text, font=("Arial", 10),
            fg="#6b7280", justify="left"
        ).pack(pady=(0, 6), anchor="w")

        start_frame = tk.LabelFrame(left, text="Start State (mot phan)", font=("Arial", 11, "bold"))
        start_frame.pack(pady=6)

        self.po_start_cells_ui = []
        for i in range(3):
            row_w = []
            for j in range(3):
                row_w.append(None)
            self.po_start_cells_ui.append(row_w)

        po_start_grid = tk.Frame(start_frame)
        po_start_grid.pack(padx=6, pady=6)
        self.po_start_grid_frame = po_start_grid

        goal_frame = tk.LabelFrame(left, text="Goal State (mot phan)", font=("Arial", 11, "bold"))
        goal_frame.pack(pady=6)

        po_goal_grid = tk.Frame(goal_frame)
        po_goal_grid.pack(padx=6, pady=6)
        self.po_goal_grid_frame = po_goal_grid

        tk.Button(
            left, text="Sinh Start/Goal moi (che 2 o)", font=("Arial", 11),
            bg="#7c3aed", fg="white", command=lambda: self._po_generate_masks(2)
        ).pack(fill="x", padx=4, pady=(0, 4))

        tk.Button(
            left, text="Sinh Start/Goal moi (che 3 o)", font=("Arial", 11),
            bg="#9333ea", fg="white", command=lambda: self._po_generate_masks(3)
        ).pack(fill="x", padx=4, pady=(0, 6))

        self._build_common_controls(run_callback=self._run_partial)

        tk.Label(self.display_frame, text="Belief-Start", font=("Arial", 11, "bold")).grid(row=0, column=0, padx=10)
        tk.Label(self.display_frame, text="Belief-Goal", font=("Arial", 11, "bold")).grid(row=0, column=1, padx=10)

        self.po_belief_start_frame = tk.Frame(self.display_frame)
        self.po_belief_start_frame.grid(row=1, column=0, padx=20, pady=8, sticky="n")

        self.po_belief_goal_frame = tk.Frame(self.display_frame)
        self.po_belief_goal_frame.grid(row=1, column=1, padx=20, pady=8, sticky="n")

        self._po_generate_masks(2)

    def _po_generate_masks(self, n_hidden):

        full_start = random_full_state()
        full_goal  = random_full_state()
        while full_goal == full_start:
            full_goal = random_full_state()

        self.po_full_start = full_start
        self.po_full_goal  = full_goal

        self.po_masked_start, self.po_hidden_start = mask_state(full_start, n_hidden)
        self.po_masked_goal,  self.po_hidden_goal  = mask_state(full_goal, n_hidden)

        self._po_draw_input_grid(
            self.po_start_grid_frame, self.po_masked_start,
            self.po_hidden_start, "po_entry_widgets_start"
        )
        self._po_draw_input_grid(
            self.po_goal_grid_frame, self.po_masked_goal,
            self.po_hidden_goal, "po_entry_widgets_goal"
        )

        self.log.delete("1.0", tk.END)
        self.path = []
        self.index = 0
        self.log_blocks = []
        self.info["text"] = ""
        self.time_label["text"] = f"Da che {n_hidden} o - hay nhap gia tri"

        for w in self.po_belief_start_frame.winfo_children():
            w.destroy()
        for w in self.po_belief_goal_frame.winfo_children():
            w.destroy()

    def _po_draw_input_grid(self, parent, masked_grid, hidden_cells, entry_dict_name):

        for w in parent.winfo_children():
            w.destroy()

        entry_dict = {}

        for i in range(3):
            for j in range(3):
                val = masked_grid[i][j]
                if val is None:
                    e = tk.Entry(
                        parent, width=3, justify="center",
                        font=("Arial", 18), bg="#fee2e2"
                    )
                    e.grid(row=i, column=j, padx=2, pady=2)
                    entry_dict[(i, j)] = e
                else:
                    lbl = tk.Label(
                        parent, width=3, height=1, font=("Arial", 18, "bold"),
                        bg="#2563eb", fg="white", relief="solid", text=str(val)
                    )
                    lbl.grid(row=i, column=j, padx=2, pady=2)

        setattr(self, entry_dict_name, entry_dict)

    def _po_read_user_values(self, entry_dict):
        result = {}
        for cell, entry in entry_dict.items():
            text = entry.get().strip()
            if not text:
                result[cell] = []
                continue
            vals = []
            for part in text.split(","):
                part = part.strip()
                if part == "":
                    continue
                try:
                    vals.append(int(part))
                except ValueError:
                    pass
            result[cell] = vals
        return result

    def _run_partial(self):

        if self.po_masked_start is None:
            messagebox.showerror("Error", "Hay sinh Start/Goal truoc")
            return

        user_start = self._po_read_user_values(self.po_entry_widgets_start)
        user_goal  = self._po_read_user_values(self.po_entry_widgets_goal)

        belief_start = generate_belief(self.po_masked_start, user_start, max_states=50)
        belief_goal  = generate_belief(self.po_masked_goal, user_goal, max_states=50)

        if not belief_start:
            messagebox.showerror("Error", "Khong sinh duoc belief-start hop le! Kiem tra gia tri da nhap.")
            return
        if not belief_goal:
            messagebox.showerror("Error", "Khong sinh duoc belief-goal hop le! Kiem tra gia tri da nhap.")
            return

        self._po_draw_belief_set(self.po_belief_start_frame, belief_start, "#2563eb")
        self._po_draw_belief_set(self.po_belief_goal_frame, belief_goal, "#16a34a")

        self.log.insert(tk.END, f"Belief-Start: {len(belief_start)} state(s)\n")
        self.log.insert(tk.END, f"Belief-Goal : {len(belief_goal)} state(s)\n\n")

        self._start_search(
            target_func=belief_state_bfs_partial,
            args=(belief_start, belief_goal),
            on_done=lambda result, elapsed: self._on_done_partial(result, belief_start, belief_goal, elapsed)
        )

    def _po_draw_belief_set(self, parent, belief, color):

        for w in parent.winfo_children():
            w.destroy()

        for idx, state in enumerate(belief):
            sf = tk.LabelFrame(parent, text=f"#{idx+1}", font=("Arial", 9))
            sf.grid(row=idx // 3, column=idx % 3, padx=4, pady=4)
            for i in range(3):
                for j in range(3):
                    val = state[i][j]
                    lbl = tk.Label(
                        sf, width=2, height=1, font=("Arial", 12, "bold"),
                        bg=color, fg="white", relief="solid",
                        text="" if val == 0 else str(val)
                    )
                    lbl.grid(row=i, column=j, padx=1, pady=1)

    def _on_done_partial(self, result, belief_start, belief_goal, elapsed):

        if result == "ALREADY_GOAL":
            self.log.insert(tk.END, "\nBelief-Start da nam trong Belief-Goal ngay tu dau! (0 buoc)\n")
            self.log.see(tk.END)
            return

        if result == "LIMIT_REACHED":
            self.log.insert(tk.END, "\n--- Vuot gioi han node, chua tim duoc solution. ---\n")
            self.log.insert(tk.END, "Hay tang 'Gioi han node' hoac sinh lai Start/Goal (it o bi che hon).\n")
            self.log.see(tk.END)
            return

        if result == "NO_SOLUTION":
            self.log.insert(tk.END, "\n--- Khong tim duoc solution (da duyet het khong gian)! ---\n")
            self.log.see(tk.END)
            return

        if result == "CANCELLED" or not result:
            self.log.insert(tk.END, "\n--- Da huy / khong co ket qua ---\n")
            self.log.see(tk.END)
            return

        self.log.delete("1.0", tk.END)
        self.log.insert(tk.END, f"Belief-Start: {len(belief_start)} state(s)\n")
        self.log.insert(tk.END, f"Belief-Goal : {len(belief_goal)} state(s)\n\n")

        self.path  = [(deepcopy(belief_start), "START")] + result
        self.index = 0

        goal_keys = set(po_po_state_to_tuple(s) for s in belief_goal)

        self.log_blocks = []
        header = (
            "===== PARTIAL OBSERVATION BFS =====\n\n"
            f"So buoc tim duoc: {len(self.path) - 1}\n\n"
        )
        for i, (belief, action) in enumerate(self.path):
            block = ""
            if i == 0:
                block += header
            block += f"STEP {i}\n"
            block += f"ACTION : {action} (belief size = {len(belief)})\n"
            n_goal = sum(1 for s in belief if po_po_state_to_tuple(s) in goal_keys)
            block += f"  Da dat belief-goal: {n_goal}/{len(belief)}\n"
            block += "-" * 35 + "\n"
            self.log_blocks.append(block)

        self._render_log_up_to(self.index)
        self._show_partial_step(belief_goal)

    def _show_partial_step(self, belief_goal):

        if not self.path:
            return

        belief, action = self.path[self.index]

        self._po_draw_belief_set(self.po_belief_start_frame, belief, "#2563eb")

        goal_keys = set(po_po_state_to_tuple(s) for s in belief_goal)
        n_goal = sum(1 for s in belief if po_po_state_to_tuple(s) in goal_keys)

        self.info["text"] = (
            f"STEP   : {self.index}/{len(self.path)-1}\n"
            f"ACTION : {action}\n"
            f"At goal: {n_goal}/{len(belief)}"
        )

        self._current_belief_goal = belief_goal

    # ==============================================================
    # 3) AND-OR-GRAPH-SEARCH
    # ==============================================================

    def _build_and_or_ui(self):

        left = self.left

        start_frame = tk.LabelFrame(left, text="Start State", font=("Arial", 11, "bold"))
        start_frame.pack(pady=6)

        self.aos_start_entries = []
        for i in range(3):
            row_e = []
            for j in range(3):
                e = tk.Entry(start_frame, width=4, justify="center", font=("Arial", 22))
                e.grid(row=i, column=j, padx=3, pady=3)
                e.insert(0, str([[1,2,3],[4,5,6],[7,0,8]][i][j]))
                row_e.append(e)
            self.aos_start_entries.append(row_e)

        goal_frame = tk.LabelFrame(left, text="Goal State", font=("Arial", 11, "bold"))
        goal_frame.pack(pady=6)

        self.aos_goal_entries = []
        for i in range(3):
            row_e = []
            for j in range(3):
                e = tk.Entry(goal_frame, width=4, justify="center", font=("Arial", 22))
                e.grid(row=i, column=j, padx=3, pady=3)
                e.insert(0, str(self.default_goal[i][j]))
                row_e.append(e)
            self.aos_goal_entries.append(row_e)

        depth_frame = tk.Frame(left)
        depth_frame.pack(fill="x", padx=4, pady=(0, 6))
        tk.Label(depth_frame, text="Max depth:", font=("Arial", 10)).pack(side="left")
        self.aos_maxdepth_var = tk.StringVar(value="20")
        tk.Entry(
            depth_frame, textvariable=self.aos_maxdepth_var,
            width=6, justify="center", font=("Arial", 10)
        ).pack(side="left", padx=6)

        self._build_common_controls(run_callback=self._run_and_or)

        self.max_nodes_var.set("50000")

        self.aos_left_cells = []
        self.aos_right_cells = []

        left_state = tk.Frame(self.display_frame)
        left_state.grid(row=0, column=0, padx=10, pady=10)
        for i in range(3):
            row_c = []
            for j in range(3):
                lbl = tk.Label(
                    left_state, width=3, height=1, font=("Arial", 20, "bold"),
                    bg="#2563eb", fg="white", relief="solid"
                )
                lbl.grid(row=i, column=j, padx=3, pady=3)
                row_c.append(lbl)
            self.aos_left_cells.append(row_c)

        tk.Label(self.display_frame, text="→", font=("Arial", 30, "bold")).grid(row=0, column=1, padx=10)

        right_state = tk.Frame(self.display_frame)
        right_state.grid(row=0, column=2, padx=10, pady=10)
        for i in range(3):
            row_c = []
            for j in range(3):
                lbl = tk.Label(
                    right_state, width=3, height=1, font=("Arial", 20, "bold"),
                    bg="#16a34a", fg="white", relief="solid"
                )
                lbl.grid(row=i, column=j, padx=3, pady=3)
                row_c.append(lbl)
            self.aos_right_cells.append(row_c)

        start = [[1,2,3],[4,5,6],[7,0,8]]
        self._draw_transition_generic(self.aos_left_cells, self.aos_right_cells, start, start)

    def _aos_read_input(self):
        try:
            start = []
            for i in range(3):
                row = []
                for j in range(3):
                    row.append(int(self.aos_start_entries[i][j].get()))
                start.append(row)
            goal = []
            for i in range(3):
                row = []
                for j in range(3):
                    row.append(int(self.aos_goal_entries[i][j].get()))
                goal.append(row)
            return start, goal
        except Exception:
            return None, None

    def _run_and_or(self):

        start, goal = self._aos_read_input()
        if start is None:
            messagebox.showerror("Error", "Invalid Input")
            return

        try:
            max_depth = int(self.aos_maxdepth_var.get())
        except ValueError:
            max_depth = 30

        self.log.insert(tk.END, "Dang chay AND-OR-GRAPH-SEARCH...\n")

        self._start_search(
            target_func=and_or_graph_search,
            args=(start, goal),
            kwargs={"max_depth": max_depth},
            on_done=lambda result, elapsed: self._on_done_and_or(result, start, goal, elapsed),
            returns_tuple=True
        )

    def _on_done_and_or(self, result, start, goal, elapsed):

        plan, expanded = result

        self.log.delete("1.0", tk.END)
        self.log.insert(tk.END, f"So node da expand (OR-SEARCH calls): {expanded}\n\n")

        if plan == "LIMIT_REACHED":
            self.log.insert(tk.END, "--- VUOT GIOI HAN NODE (max_expanded) - chua tim duoc plan ---\n")
            self.log.insert(tk.END, "Hay tang 'Gioi han node' hoac giam Max depth.\n")
            self.log.see(tk.END)
            return

        if plan == "failure" or plan is None:
            self.log.insert(tk.END, "--- KHONG TIM DUOC PLAN (failure / vuot max depth) ---\n")
            self.log.see(tk.END)
            return

        sequence = flatten_plan(plan, start)

        self.path = sequence
        self.index = 0

        self.log_blocks = []
        header = (
            "===== AND-OR-GRAPH-SEARCH =====\n\n"
            f"So node da expand: {expanded}\n"
            f"So buoc trong plan: {len(sequence) - 1}\n\n"
        )
        for i, (state, action) in enumerate(sequence):
            block = ""
            if i == 0:
                block += header
            block += f"STEP {i}\n"
            block += f"ACTION : {action}\n"
            rows = " | ".join(
                " ".join(str(v) if v != 0 else "_" for v in state[r])
                for r in range(3)
            )
            block += f"  STATE: {rows}\n"
            block += "-" * 35 + "\n"
            self.log_blocks.append(block)

        self._render_log_up_to(self.index)
        self._show_and_or_step()

    def _show_and_or_step(self):

        if not self.path:
            return

        state, action = self.path[self.index]

        if self.index < len(self.path) - 1:
            next_state, _ = self.path[self.index + 1]
        else:
            next_state = state

        self._draw_transition_generic(self.aos_left_cells, self.aos_right_cells, state, next_state)

        self.info["text"] = (
            f"STEP   : {self.index}/{len(self.path)-1}\n"
            f"ACTION : {action}"
        )

    # ==============================================================
    # COMMON CONTROLS
    # ==============================================================

    def _build_common_controls(self, run_callback):

        left = self.left

        control = tk.Frame(left)
        control.pack(pady=8)

        btn_cfg = dict(width=10, font=("Arial", 11))

        self._run_callback = run_callback

        tk.Button(control, text="Run", command=self._on_run_click, **btn_cfg
                  ).grid(row=0, column=0, padx=4, pady=3)
        tk.Button(control, text="Reset", command=self._reset, **btn_cfg
                  ).grid(row=0, column=1, padx=4, pady=3)
        tk.Button(control, text="Prev", command=self._prev_step, **btn_cfg
                  ).grid(row=1, column=0, padx=4, pady=3)
        tk.Button(control, text="Next", command=self._next_step, **btn_cfg
                  ).grid(row=1, column=1, padx=4, pady=3)
        tk.Button(control, text="Auto Run", command=self._auto_run,
                  width=22, font=("Arial", 11)
                  ).grid(row=2, column=0, columnspan=2, pady=3)

        self.btn_cancel = tk.Button(
            control, text="Cancel", command=self._cancel,
            width=22, font=("Arial", 11), bg="#dc2626", fg="white",
            state="disabled"
        )
        self.btn_cancel.grid(row=3, column=0, columnspan=2, pady=3)

        self.speed = tk.Scale(left, from_=100, to=2000, orient="horizontal", label="Speed (ms)")
        self.speed.set(600)
        self.speed.pack(fill="x", padx=4)

        limit_frame = tk.Frame(left)
        limit_frame.pack(fill="x", padx=4, pady=(6, 0))
        tk.Label(limit_frame, text="Gioi han node:", font=("Arial", 10)).pack(side="left")
        self.max_nodes_var = tk.StringVar(value="5000")
        tk.Entry(limit_frame, textvariable=self.max_nodes_var, width=8, justify="center", font=("Arial", 10)
                 ).pack(side="left", padx=6)

        self.info = tk.Label(left, text="", justify="left", anchor="w", font=("Consolas", 11))
        self.info.pack(fill="x", padx=4, pady=4)

        self.time_label = tk.Label(left, text="", justify="left", anchor="w", font=("Consolas", 10), fg="#6b7280")
        self.time_label.pack(fill="x", padx=4)

    def _on_run_click(self):
        if self._running:
            return
        self.log.delete("1.0", tk.END)
        self._run_callback()

    # ==============================================================
    # GENERIC DRAW HELPERS
    # ==============================================================

    def _draw_belief_generic(self, cells, belief, prev_belief=None):

        CHANGED_BG = "#dc2626"
        COLORS     = ["#2563eb", "#7c3aed", "#0891b2", "#ca8a04", "#059669"]

        for b in range(len(belief)):
            for i in range(3):
                for j in range(3):
                    val = belief[b][i][j]
                    changed = (
                        prev_belief is not None and
                        b < len(prev_belief) and
                        belief[b][i][j] != prev_belief[b][i][j]
                    )
                    cells[b][i][j]["text"] = "" if val == 0 else str(val)
                    cells[b][i][j]["bg"] = CHANGED_BG if changed else COLORS[b % len(COLORS)]

    def _draw_transition_generic(self, left_cells, right_cells, cur, nxt):

        for i in range(3):
            for j in range(3):
                a = cur[i][j]
                b = nxt[i][j]
                left_cells[i][j]["text"]  = "" if a == 0 else str(a)
                right_cells[i][j]["text"] = "" if b == 0 else str(b)

    # ==============================================================
    # SEARCH RUNNER
    # ==============================================================

    def _start_search(self, target_func, args, on_done, kwargs=None, returns_tuple=False):

        if self._running:
            return

        kwargs = kwargs or {}

        self._running = True
        self._stop_event.clear()
        self.btn_cancel.config(state="normal")
        self.time_label.config(text="Dang tim kiem...")

        while not self._log_queue.empty():
            try: self._log_queue.get_nowait()
            except Exception: pass

        try:
            max_n = int(self.max_nodes_var.get())
            if max_n < 1:
                max_n = 5000
        except ValueError:
            max_n = 5000

        def _worker():
            t_start = time.perf_counter()

            call_kwargs = dict(kwargs)

            sig = inspect.signature(target_func)
            if "stop_event" in sig.parameters:
                call_kwargs["stop_event"] = self._stop_event
            if "log_queue" in sig.parameters:
                call_kwargs["log_queue"] = self._log_queue
            if "max_nodes" in sig.parameters:
                call_kwargs["max_nodes"] = max_n
            if "max_expanded" in sig.parameters:
                call_kwargs["max_expanded"] = max_n

            result = target_func(*args, **call_kwargs)
            elapsed = time.perf_counter() - t_start

            self._log_queue.put(("__DONE__", result, elapsed))

        threading.Thread(target=_worker, daemon=True).start()
        self._current_on_done = on_done
        self._poll_log()

    def _poll_log(self):

        count = 0
        while count < 50:
            try:
                item = self._log_queue.get_nowait()
            except Queue.Empty:
                break

            if isinstance(item, tuple) and item[0] == "__DONE__":
                _, result, elapsed = item
                self._running = False
                self.btn_cancel.config(state="disabled")

                if self._stop_event.is_set():
                    self.time_label.config(text=f"Da huy sau {elapsed:.3f}s")
                    self.log.insert(tk.END, "\n--- Da huy tim kiem ---\n")
                    self.log.see(tk.END)
                    return

                self.time_label.config(text=f"Thoi gian: {elapsed:.4f}s")
                self._current_on_done(result, elapsed)
                return

            if isinstance(item, str) and item.startswith("__LIMIT__:"):
                n = item.split(":")[1]
                self.log.insert(tk.END, f"\n--- Dung: da expand {n} nodes (vuot gioi han) ---\n")
                self.log.see(tk.END)
                continue

            self.log.insert(tk.END, item)
            self.log.see(tk.END)
            count += 1

        self._poll_id = self.root.after(80, self._poll_log)

    def _cancel(self):
        self._stop_event.set()

    # ==============================================================
    # LOG + NAVIGATION
    # ==============================================================

    def _render_log_up_to(self, idx):
        self.log.delete("1.0", tk.END)
        for i in range(min(idx + 1, len(self.log_blocks))):
            self.log.insert(tk.END, self.log_blocks[i])
        self.log.see(tk.END)

    def _show_current_step(self):
        algo = self.algo_var.get()
        if algo == ALGO_BELIEF_BFS:
            self._show_belief_bfs_step()
        elif algo == ALGO_PARTIAL:
            self._show_partial_step(getattr(self, "_current_belief_goal", []))
        elif algo == ALGO_AND_OR:
            self._show_and_or_step()

    def _next_step(self):
        if self.index < len(self.path) - 1:
            self.index += 1
            self._show_current_step()
            self._render_log_up_to(self.index)

    def _prev_step(self):
        if self.index > 0:
            self.index -= 1
            self._show_current_step()
            self._render_log_up_to(self.index)

    def _auto_run(self):
        if not self.path:
            return
        self.index = 0
        self._animate()

    def _animate(self):
        if self.index >= len(self.path):
            return
        self._show_current_step()
        self._render_log_up_to(self.index)
        self.index += 1
        if self.index < len(self.path):
            self.root.after(self.speed.get(), self._animate)

    def _reset(self):
        self.path = []
        self.index = 0
        self.log_blocks = []
        self.info["text"] = ""
        self.time_label["text"] = ""
        self.log.delete("1.0", tk.END)
        self._switch_algo()


# ==========================================
# MAIN
# ==========================================

root = tk.Tk()
MultiAlgoGUI(root)
root.mainloop()